# Dermavant — SCIN common-condition triage baseline

**Purpose:** train an image-based triage classifier for common skin concerns. SCIN labels are retrospective dermatologist differentials, not confirmed diagnoses. We therefore predict broad triage categories and retain an `uncertain` route.

**Never use this notebook with identifiable Dermavant patient photos in Kaggle.**

## Kaggle setup

Attach only `nikita2998/scin-dataset`, set Accelerator to **GPU T4 x2**, and enable Internet only for the pretrained EfficientNet weights. Run the dependency cell, then use **Restart Session** before continuing. The first schema cell intentionally fails if this mirror does not preserve the official `scin_cases.csv`, `scin_labels.csv`, and image paths.

In [ ]:
# Pin Pillow exactly: Kaggle's torchvision build can fail with newer Pillow releases.
# After this cell completes, select Run > Restart Session, then resume at the next cell.
!pip -q install --no-cache-dir --force-reinstall "Pillow==10.4.0"
!pip -q install --no-cache-dir "timm==1.0.15" "scikit-learn==1.5.2" "albumentations==1.4.18"


In [ ]:
from pathlib import Path
import ast, json, random, re
from collections import Counter
import numpy as np
import pandas as pd
from PIL import Image
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from tqdm.auto import tqdm
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

SEED, IMAGE_SIZE, BATCH_SIZE_PER_GPU, EPOCHS = 20260907, 224, 32, 20
EVAL_BATCH_SIZE_PER_GPU = 128
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu'); N_GPUS = torch.cuda.device_count()
# `/kaggle/input` is read-only. Save all checkpoints and reports only under `/kaggle/working`.
OUTPUT_DIR = Path('/kaggle/working/dermavant_scin_v1')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
write_test = OUTPUT_DIR / '.write_test'
write_test.write_text('ok'); write_test.unlink()
assert str(OUTPUT_DIR).startswith('/kaggle/working/'), f'Checkpoint output must be under /kaggle/working, not {OUTPUT_DIR}'
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED); torch.backends.cudnn.benchmark = True
assert DEVICE.type == 'cuda', 'Enable Kaggle GPU before training.'
print({'device':str(DEVICE),'gpu_count':N_GPUS,'output_dir':str(OUTPUT_DIR)})


In [ ]:
# Kaggle's SCIN mirror prefixes the official file names with `dataset_`.
ROOT = Path('/kaggle/input/datasets/nikita2998/scin-dataset/SCIN-dataset')
assert ROOT.exists(), f'Attach the exact SCIN Kaggle dataset: {ROOT}'
csvs = {p.name: p for p in ROOT.rglob('*.csv')}
print('CSV files:', list(csvs)); print('images:', len(list(ROOT.rglob('*.jpg'))) + len(list(ROOT.rglob('*.jpeg'))) + len(list(ROOT.rglob('*.png'))))
CASE_CSV = next((name for name in ['scin_cases.csv', 'dataset_scin_cases.csv'] if name in csvs), None)
LABEL_CSV = next((name for name in ['scin_labels.csv', 'dataset_scin_labels.csv'] if name in csvs), None)
assert CASE_CSV and LABEL_CSV, f'Expected SCIN cases and labels files. Found: {list(csvs)}'
cases, labels = pd.read_csv(csvs[CASE_CSV]), pd.read_csv(csvs[LABEL_CSV])
required_cases = {'case_id'}; required_labels = {'case_id','weighted_skin_condition_label'}
assert required_cases.issubset(cases.columns) and required_labels.issubset(labels.columns), f'SCIN schema is not compatible. Cases: {cases.columns.tolist()}; labels: {labels.columns.tolist()}'
image_columns = [c for c in cases.columns if re.fullmatch(r'image_[1-3]_path', c)]
assert image_columns, 'Expected image_1_path through image_3_path.'
print({'case_file':CASE_CSV, 'label_file':LABEL_CSV, 'cases':len(cases),'labels':len(labels),'image_columns':image_columns})


In [ ]:
# Map the official differential labels into safe product-routing classes. A clinician must approve this map before release.
TRIAGE_MAP = {
 'eczema': 'inflammatory_allergic', 'dermatitis': 'inflammatory_allergic', 'urticaria': 'inflammatory_allergic', 'psoriasis': 'inflammatory_allergic',
 'acne': 'inflammatory_allergic', 'rosacea': 'inflammatory_allergic',
 'tinea': 'possible_infectious', 'fungal': 'possible_infectious', 'impetigo': 'possible_infectious', 'scabies': 'possible_infectious', 'cellulitis': 'possible_infectious',
 'wart': 'other_or_growth', 'nevus': 'other_or_growth', 'melanoma': 'other_or_growth', 'carcinoma': 'other_or_growth', 'mole': 'other_or_growth',
 'alopecia': 'hair_or_nail', 'hair': 'hair_or_nail', 'nail': 'hair_or_nail',
}
CLASS_NAMES = ['inflammatory_allergic','possible_infectious','other_or_growth','hair_or_nail','uncertain']
CLASS_TO_ID = {name:i for i,name in enumerate(CLASS_NAMES)}

def parse_weighted_label(value):
    if not isinstance(value, str) or not value.strip(): return {}
    try: return ast.literal_eval(value)
    except (ValueError, SyntaxError): return {}

def triage_label(weighted):
    labels_dict = parse_weighted_label(weighted)
    if not labels_dict: return 'uncertain'
    condition, weight = max(labels_dict.items(), key=lambda item: item[1])
    if float(weight) < 0.55: return 'uncertain'
    normalized = condition.lower()
    return next((category for key, category in TRIAGE_MAP.items() if key in normalized), 'uncertain')

# Retain only cases a dermatologist marked as having sufficient image quality when those columns are available.
gradable_cols = [c for c in labels.columns if c.startswith('dermatologist_gradable_for_skin_condition_')]
if gradable_cols:
    def acceptable(values):
        allowed = {'TRUE', 'YES', 'DEFAULT_YES_IMAGE_QUALITY_SUFFICIENT'}
        return any((value is True) or (str(value).upper() in allowed) for value in values)
    labels = labels.loc[labels[gradable_cols].apply(lambda row: acceptable(row.tolist()), axis=1)].copy()
frame = cases.merge(labels[['case_id','weighted_skin_condition_label']], on='case_id', how='inner')
frame['triage_class'] = frame.weighted_skin_condition_label.map(triage_label)
print(frame.triage_class.value_counts())
assert (frame.triage_class != 'uncertain').sum() > 100, 'Too few mapped cases. Review the SCIN mirror and clinician mapping.'


In [ ]:
# Expand one case into up to three images. Only retain readable, locally present image files.
all_images = list(ROOT.rglob('*.jpg')) + list(ROOT.rglob('*.jpeg')) + list(ROOT.rglob('*.png'))
by_name = {p.name:p for p in all_images}; by_relative = {str(p.relative_to(ROOT)).replace('\\','/'):p for p in all_images}
records = []
for row in frame.itertuples(index=False):
    row_dict = row._asdict()
    for col in image_columns:
        source_path = row_dict.get(col)
        if not isinstance(source_path, str) or not source_path: continue
        path = by_relative.get(source_path.replace('\\','/')) or by_name.get(Path(source_path).name)
        if path: records.append({'case_id':row_dict['case_id'],'image_path':str(path),'triage_class':row_dict['triage_class'], 'fitzpatrick_skin_type':row_dict.get('fitzpatrick_skin_type', None), 'monk_skin_tone':row_dict.get('monk_skin_tone_label_us', None)})
images = pd.DataFrame(records).drop_duplicates('image_path')
assert len(images) > 1000, f'Only {len(images)} images mapped; stop and inspect image paths.'
print(images.triage_class.value_counts(), '\nTotal images:',len(images),'cases:',images.case_id.nunique())

# Split by case to keep all images from an individual out of validation/test.
splitter = GroupShuffleSplit(n_splits=1, test_size=.30, random_state=SEED)
tr_idx, ho_idx = next(splitter.split(images, groups=images.case_id)); train_df, holdout = images.iloc[tr_idx].copy(), images.iloc[ho_idx].copy()
splitter = GroupShuffleSplit(n_splits=1, test_size=.50, random_state=SEED+1)
va_idx, te_idx = next(splitter.split(holdout, groups=holdout.case_id)); val_df, test_df = holdout.iloc[va_idx].copy(), holdout.iloc[te_idx].copy()
for name, part in [('train',train_df),('validation',val_df),('test',test_df)]: print(name, len(part), part.case_id.nunique(), part.triage_class.value_counts().to_dict())


In [ ]:
train_tfms = A.Compose([A.RandomResizedCrop(size=(IMAGE_SIZE,IMAGE_SIZE),scale=(.75,1.0)), A.HorizontalFlip(p=.5), A.Rotate(limit=12,p=.4), A.RandomBrightnessContrast(.12,.12,p=.3), A.Normalize(), ToTensorV2()])
eval_tfms = A.Compose([A.Resize(IMAGE_SIZE,IMAGE_SIZE), A.Normalize(), ToTensorV2()])

class SCINDataset(Dataset):
 def __init__(self, frame, tfms): self.paths=frame.image_path.tolist(); self.labels=frame.triage_class.tolist(); self.cases=frame.case_id.tolist(); self.tfms=tfms
 def __len__(self): return len(self.paths)
 def __getitem__(self, idx):
  with Image.open(self.paths[idx]) as source: image=np.asarray(source.convert('RGB'))
  return self.tfms(image=image)['image'], torch.tensor(CLASS_TO_ID[self.labels[idx]]), self.cases[idx]

def make_loader(frame, train=False):
 ds=SCINDataset(frame, train_tfms if train else eval_tfms); per_gpu=BATCH_SIZE_PER_GPU if train else EVAL_BATCH_SIZE_PER_GPU
 if train:
  counts=frame.triage_class.value_counts(); weights=frame.triage_class.map(lambda x: 1/counts[x]).to_numpy(); sampler=WeightedRandomSampler(weights,len(weights),replacement=True)
  return DataLoader(ds,batch_size=per_gpu*max(1,N_GPUS),sampler=sampler,num_workers=4,pin_memory=True,persistent_workers=True,prefetch_factor=4)
 return DataLoader(ds,batch_size=per_gpu*max(1,N_GPUS),shuffle=False,num_workers=4,pin_memory=True,persistent_workers=True,prefetch_factor=4)
train_loader,val_loader,test_loader=make_loader(train_df,True),make_loader(val_df),make_loader(test_df)
print({'validation_batches':len(val_loader),'test_batches':len(test_loader),'eval_batch_size':EVAL_BATCH_SIZE_PER_GPU*max(1,N_GPUS)})


In [ ]:
model=timm.create_model('efficientnet_b0',pretrained=True,num_classes=len(CLASS_NAMES)).to(DEVICE)
if N_GPUS>1: model=nn.DataParallel(model)
criterion=nn.CrossEntropyLoss(label_smoothing=.05); optimizer=torch.optim.AdamW(model.parameters(),lr=2e-4,weight_decay=1e-4); scheduler=torch.optim.lr_scheduler.CosineAnnealingLR(optimizer,T_max=EPOCHS); scaler=torch.amp.GradScaler('cuda')

@torch.inference_mode()
def evaluate(loader, description='Evaluation'):
 model.eval(); probs=[]; targets=[]; cases=[]
 for x,y,case in tqdm(loader,desc=description,leave=False):
  with torch.amp.autocast('cuda'): logits=model(x.to(DEVICE,non_blocking=True))
  probs.append(torch.softmax(logits,1).cpu().numpy()); targets.extend(y.numpy()); cases.extend(case)
 return np.concatenate(probs),np.asarray(targets),cases

best=-1; history=[]
for epoch in range(1,EPOCHS+1):
 model.train(); total=0
 for x,y,_ in tqdm(train_loader,desc=f'Training epoch {epoch}/{EPOCHS}',leave=False):
  optimizer.zero_grad(set_to_none=True)
  with torch.amp.autocast('cuda'): loss=criterion(model(x.to(DEVICE,non_blocking=True)),y.to(DEVICE,non_blocking=True))
  scaler.scale(loss).backward(); scaler.step(optimizer); scaler.update(); total+=loss.item()*len(x)
 scheduler.step(); p,y,_=evaluate(val_loader,f'Validation epoch {epoch}/{EPOCHS}'); score=f1_score(y,p.argmax(1),average='macro',zero_division=0); rec={'epoch':epoch,'loss':total/len(train_df),'val_macro_f1':score}; history.append(rec); print(rec)
 if score>best:
  best=score; torch.save({'model_state':(model.module if isinstance(model,nn.DataParallel) else model).state_dict(),'epoch':epoch,'val_macro_f1':score,'class_names':CLASS_NAMES},OUTPUT_DIR/'best.pt')
pd.DataFrame(history).to_csv(OUTPUT_DIR/'training_history.csv',index=False)


In [ ]:
base=model.module if isinstance(model,nn.DataParallel) else model; checkpoint=torch.load(OUTPUT_DIR/'best.pt',map_location=DEVICE); base.load_state_dict(checkpoint['model_state'])
probs,targets,cases=evaluate(test_loader); predicted=probs.argmax(1)
print(classification_report(targets,predicted,target_names=CLASS_NAMES,zero_division=0))
pd.DataFrame(confusion_matrix(targets,predicted),index=CLASS_NAMES,columns=CLASS_NAMES)

# Fairness reporting only: no race or skin-tone feature is sent into the model.
result=test_df.reset_index(drop=True)[['case_id','triage_class','fitzpatrick_skin_type','monk_skin_tone']].copy(); result['prediction']=[CLASS_NAMES[x] for x in predicted]; result['confidence']=probs.max(1); result['correct']=(targets==predicted)
if result.fitzpatrick_skin_type.notna().any(): print(result.groupby('fitzpatrick_skin_type').correct.agg(['count','mean']).sort_index())
if result.monk_skin_tone.notna().any(): print(result.groupby('monk_skin_tone').correct.agg(['count','mean']).sort_index())
result.to_csv(OUTPUT_DIR/'heldout_predictions_and_fairness_audit.csv',index=False)
torch.save({'model_state':base.state_dict(),'class_names':CLASS_NAMES,'dataset':'SCIN','intended_use':'assistive common-condition triage only; not diagnostic','license_note':'Preserve SCIN attribution and privacy restrictions'},OUTPUT_DIR/'dermavant_scin_common_triage_v1.pt')
print('Exported:',list(OUTPUT_DIR.iterdir()))


## Release gate

A dermatologist must approve the category mapping, held-out errors, and skin-tone subgroup report before deployment. The `uncertain` class is a safety feature: it must route to clinician review, not be hidden from users.